# TrustCV spatial methods with UniversalCVRunner `split_kwargs`

This Colab-ready notebook tests the four spatial methods in the TrustCV coverage matrix using the updated `UniversalCVRunner.run()` API:

```python
runner.run(..., split_kwargs={...}, fit_kwargs={...}, evaluate_kwargs={...})
```

The notebook no longer uses a notebook-level `BoundTrustCVSplitter` helper. Spatial metadata such as `coordinates`, `timestamps`, and `environmental_data` are passed only to the TrustCV splitter through `split_kwargs`, while model-fitting arguments stay separated in `fit_kwargs`.

The example generates synthetic environmental-health data and exercises:

- `SpatialBlockCV`
- `BufferedSpatialCV`
- `SpatiotemporalBlockCV`
- `EnvironmentalHealthCV`
- `UniversalCVRunner`
- `CVResults`
- `LeakageDetectionCallback`
- `ClassDistributionLogger`
- `DataLeakageChecker`
- `ClinicalMetrics`
- `oob_clinical_metrics`

The goal is to verify that the toolbox can run spatial and spatiotemporal splitters through the same runner interface used by the rest of TrustCV.

## Cell 1 ? Runtime settings

Use `SMOKE_MODE=True` first. After the notebook runs cleanly in Colab, set `SMOKE_MODE=False` for the final repository example.

For local development inside this repository, keep `INSTALL_SOURCE="none"` so the notebook imports the checked-out TrustCV code. Use `github` or `pypi` only after the updated `split_kwargs` API is available from that source.


In [ ]:
import sys
from pathlib import Path

RUNNING_IN_COLAB = "google.colab" in sys.modules

# Use "none" for local repository runs so the notebook imports this checkout.
# Use "github" only after the split_kwargs API has been pushed to the target branch.
# Use "pypi" only after the split_kwargs API is released to PyPI.
INSTALL_SOURCE = "none"  # options: "none", "github", "pypi"
TRUSTCV_GITHUB_URL = "git+https://github.com/ki-smile/trustcv.git"

SMOKE_MODE = True
RANDOM_STATE = 42
OUTPUT_DIR = (
    "/content/trustcv_spatial_split_kwargs_outputs"
    if RUNNING_IN_COLAB
    else str(Path.cwd() / "outputs" / "trustcv_spatial_split_kwargs_outputs")
)

# Synthetic data size
N_SITES = 24 if SMOKE_MODE else 80
N_MONTHS = 10 if SMOKE_MODE else 24
N_ESTIMATORS = 80 if SMOKE_MODE else 200

# Spatial validation configuration
N_SPATIAL_SPLITS = 4 if SMOKE_MODE else 5
N_SPATIAL_BLOCKS = 2 if SMOKE_MODE else 3
N_TEMPORAL_BLOCKS = 2 if SMOKE_MODE else 4

# Buffers are expressed in synthetic coordinate units, because coordinates are generated in [0, 1] x [0, 1].
BUFFER_SIZE = 0.10 if SMOKE_MODE else 0.08
SPATIOTEMPORAL_BUFFER_SPACE = 0.03 if SMOKE_MODE else 0.04
ENV_BUFFER_CONFIG = {"pm25": 0.30, "no2": 0.50} if SMOKE_MODE else {"pm25": 0.50, "no2": 0.75}

print({
    "INSTALL_SOURCE": INSTALL_SOURCE,
    "SMOKE_MODE": SMOKE_MODE,
    "N_SITES": N_SITES,
    "N_MONTHS": N_MONTHS,
    "N_SPATIAL_SPLITS": N_SPATIAL_SPLITS,
    "N_SPATIAL_BLOCKS": N_SPATIAL_BLOCKS,
    "N_TEMPORAL_BLOCKS": N_TEMPORAL_BLOCKS,
})


## Cell 2 ? Install or select TrustCV

For local development, this cell selects the repository checkout instead of reinstalling TrustCV. In Colab, switch `INSTALL_SOURCE` to `github` only after the fixed branch is pushed.


In [ ]:
import subprocess
import sys
from pathlib import Path


def pip_install(*packages):
    cmd = [sys.executable, "-m", "pip", "install", "-q", *packages]
    print("Running:", " ".join(cmd))
    subprocess.check_call(cmd)


def find_local_trustcv_repo():
    candidates = [Path.cwd(), *Path.cwd().parents]
    candidates += [Path("/content/TrustCV_Repo"), Path("/content/trustcv")]
    for candidate in candidates:
        if (candidate / "trustcv" / "core" / "runner.py").exists():
            return candidate.resolve()
    return None


if INSTALL_SOURCE == "github":
    pip_install(TRUSTCV_GITHUB_URL, "scikit-learn>=1.3,<1.8", "pandas", "matplotlib", "tabulate")
elif INSTALL_SOURCE == "pypi":
    pip_install("trustcv>=1.0.7", "scikit-learn>=1.3,<1.8", "pandas", "matplotlib", "tabulate")
elif INSTALL_SOURCE == "none":
    repo_root = find_local_trustcv_repo()
    if repo_root is None:
        raise RuntimeError(
            "INSTALL_SOURCE=none, but no local TrustCV checkout was found. "
            "Run the notebook from the repository root, or set INSTALL_SOURCE to a fixed GitHub/PyPI source."
        )
    repo_root_str = str(repo_root)
    if repo_root_str not in sys.path:
        sys.path.insert(0, repo_root_str)
    for module_name in list(sys.modules):
        if module_name == "trustcv" or module_name.startswith("trustcv."):
            del sys.modules[module_name]
    print("Using local TrustCV checkout:", repo_root)
else:
    raise ValueError("INSTALL_SOURCE must be 'none', 'github', or 'pypi'.")

import trustcv
print("trustcv version:", getattr(trustcv, "__version__", "unknown"))
print("trustcv path:", trustcv.__file__)


## Cell 3 — Imports, TrustCV API inventory, and runner API check

This cell imports the TrustCV functions used in the notebook and verifies that your local toolbox has the new `UniversalCVRunner.run()` arguments. If this cell raises an error, the notebook is running against an older TrustCV build and should be pointed to your fixed branch/package.

In [ ]:
from pathlib import Path
import inspect
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Make this cell robust when run out of order or after an old TrustCV import.
RUNNING_IN_COLAB = globals().get("RUNNING_IN_COLAB", "google.colab" in sys.modules)
INSTALL_SOURCE = globals().get("INSTALL_SOURCE", "none")
OUTPUT_DIR = globals().get(
    "OUTPUT_DIR",
    "/content/trustcv_spatial_split_kwargs_outputs"
    if RUNNING_IN_COLAB
    else str(Path.cwd() / "outputs" / "trustcv_spatial_split_kwargs_outputs"),
)


def find_local_trustcv_repo_for_import():
    candidates = [Path.cwd(), *Path.cwd().parents]
    candidates += [Path("/content/TrustCV_Repo"), Path("/content/trustcv")]
    for candidate in candidates:
        if (candidate / "trustcv" / "core" / "runner.py").exists():
            return candidate.resolve()
    return None


if INSTALL_SOURCE == "none":
    repo_root = find_local_trustcv_repo_for_import()
    if repo_root is None:
        raise RuntimeError(
            "INSTALL_SOURCE=none, but no local TrustCV checkout was found. "
            "Run the notebook from the repository root or run the install/select cell first."
        )
    repo_root_str = str(repo_root)
    if repo_root_str not in sys.path:
        sys.path.insert(0, repo_root_str)
    for module_name in list(sys.modules):
        if module_name == "trustcv" or module_name.startswith("trustcv."):
            del sys.modules[module_name]

from trustcv import (
    SpatialBlockCV,
    BufferedSpatialCV,
    SpatiotemporalBlockCV,
    EnvironmentalHealthCV,
    UniversalCVRunner,
    CVResults,
    LeakageDetectionCallback,
    ClassDistributionLogger,
    DataLeakageChecker,
    ClinicalMetrics,
)
from trustcv.metrics import oob_clinical_metrics
import trustcv

# sklearn-compatible estimator evaluated by TrustCV.
# Cross-validation, fold construction, leakage checking, and clinical metrics are TrustCV.
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

warnings.filterwarnings("ignore")
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
print("trustcv path:", trustcv.__file__)
print("OUTPUT_DIR:", OUTPUT_DIR)

run_signature = inspect.signature(UniversalCVRunner.run)
required_runner_args = {"split_kwargs", "fit_kwargs", "evaluate_kwargs"}
missing_runner_args = required_runner_args.difference(run_signature.parameters)
if missing_runner_args:
    local_hint = find_local_trustcv_repo_for_import()
    raise RuntimeError(
        "This notebook requires the updated UniversalCVRunner.run API. "
        f"Missing arguments: {sorted(missing_runner_args)}. "
        f"Imported trustcv from: {trustcv.__file__}. "
        f"Local checkout detected: {local_hint}. "
        "Set INSTALL_SOURCE='none' and rerun this cell, or restart the kernel and rerun from Cell 1."
    )

api_rows = []
for obj in [
    SpatialBlockCV,
    BufferedSpatialCV,
    SpatiotemporalBlockCV,
    EnvironmentalHealthCV,
    UniversalCVRunner,
    CVResults,
    LeakageDetectionCallback,
    ClassDistributionLogger,
    DataLeakageChecker,
    ClinicalMetrics,
    oob_clinical_metrics,
]:
    api_rows.append({
        "TrustCV object": getattr(obj, "__name__", str(obj)),
        "Signature": str(inspect.signature(obj)) if callable(obj) else "n/a",
        "Module": getattr(obj, "__module__", "n/a"),
    })

api_inventory = pd.DataFrame(api_rows)
display(api_inventory)

print("UniversalCVRunner.run signature:")
print(run_signature)


## Cell 4 — New runner pattern: splitter metadata goes into `split_kwargs`

Spatial and environmental-health splitters need metadata that normal estimators should not receive. The updated TrustCV runner solves this by separating arguments:

- `split_kwargs`: passed only to `splitter.split(...)`
- `fit_kwargs`: passed only to model fitting
- `evaluate_kwargs`: reserved for evaluation-only arguments

This notebook uses that pattern directly, so no notebook-level wrapper or bound splitter is needed.

In [ ]:
# This small smoke object is only for documentation inside the notebook.
# The actual split_kwargs for each method are created in Cell 9.
runner_argument_pattern = pd.DataFrame([
    {
        "Argument": "split_kwargs",
        "Used by": "TrustCV splitter only",
        "Examples": "coordinates, timestamps, environmental_data",
    },
    {
        "Argument": "fit_kwargs",
        "Used by": "model.fit / adapter.train_epoch only",
        "Examples": "sample_weight if needed",
    },
    {
        "Argument": "evaluate_kwargs",
        "Used by": "evaluation only, if the adapter supports it",
        "Examples": "reserved for future evaluation metadata",
    },
])
display(runner_argument_pattern)

## Cell 5 — Synthetic environmental-health data generator

The synthetic data are designed to mimic an environmental-health setting: repeated monthly observations at monitoring sites, spatially autocorrelated pollution, seasonal variation, missing exposures, and a binary respiratory-health outcome.

In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


def generate_synthetic_environmental_health_data(
    n_sites=N_SITES,
    n_months=N_MONTHS,
    random_state=RANDOM_STATE,
):
    rng = np.random.default_rng(random_state)

    # Fixed monitoring-site coordinates in a unit square.
    site_coordinates = rng.uniform(0.02, 0.98, size=(n_sites, 2))

    rows = []
    start = pd.Timestamp("2020-01-01")
    for site_id in range(n_sites):
        x_coord, y_coord = site_coordinates[site_id]
        site_risk = rng.normal(0.0, 0.25)

        for month_index in range(n_months):
            timestamp = start + pd.DateOffset(months=int(month_index))
            season = np.sin(2 * np.pi * month_index / 12.0)

            # Spatial hotspots plus seasonal exposure.
            hotspot_1 = np.exp(-((x_coord - 0.72) ** 2 + (y_coord - 0.35) ** 2) / 0.035)
            hotspot_2 = np.exp(-((x_coord - 0.25) ** 2 + (y_coord - 0.78) ** 2) / 0.055)
            pm25 = 8 + 9 * hotspot_1 + 4 * hotspot_2 + 2.0 * season + rng.normal(0, 1.1)
            no2 = 16 + 7 * (1 - x_coord) + 2.5 * hotspot_1 + rng.normal(0, 1.8)
            temperature = 13 + 11 * season + rng.normal(0, 1.3)
            humidity = 52 + 10 * y_coord - 4 * season + rng.normal(0, 3.0)
            greenspace = np.clip(0.70 - 0.45 * hotspot_1 + 0.15 * rng.normal(), 0, 1)

            # Health event probability. The task is intentionally not too easy.
            linear_risk = (
                -4.2
                + 0.12 * pm25
                + 0.035 * no2
                + 0.010 * humidity
                - 0.55 * greenspace
                + 0.30 * season
                + 0.55 * hotspot_1
                + site_risk
            )
            event_probability = sigmoid(linear_risk)
            outcome = int(rng.random() < event_probability)

            rows.append({
                "site_id": site_id,
                "month_index": month_index,
                "timestamp": timestamp,
                "x": x_coord,
                "y": y_coord,
                "pm25": pm25,
                "no2": no2,
                "temperature": temperature,
                "humidity": humidity,
                "greenspace": greenspace,
                "respiratory_event": outcome,
            })

    df = pd.DataFrame(rows)

    # Add small missingness to mimic real exposure tables.
    for col, frac in {"pm25": 0.04, "no2": 0.03, "humidity": 0.02}.items():
        mask = rng.random(len(df)) < frac
        df.loc[mask, col] = np.nan

    return df

## Cell 6 — Generate and describe the synthetic data

The model features are ordinary exposure and covariate columns. Coordinates, timestamps, and environmental arrays are passed separately to TrustCV splitters.

In [ ]:
df = generate_synthetic_environmental_health_data()

FEATURE_COLUMNS = ["pm25", "no2", "temperature", "humidity", "greenspace", "x", "y", "month_index"]
TARGET_COLUMN = "respiratory_event"

X = df[FEATURE_COLUMNS].to_numpy()
y = df[TARGET_COLUMN].to_numpy().astype(int)
groups = df["site_id"].to_numpy()
coordinates = df[["x", "y"]].to_numpy()
timestamps = pd.DatetimeIndex(pd.to_datetime(df["timestamp"]))
environmental_data = {
    "pm25": df["pm25"].fillna(df["pm25"].median()).to_numpy(),
    "no2": df["no2"].fillna(df["no2"].median()).to_numpy(),
}

summary = pd.DataFrame({
    "quantity": [
        "n_rows",
        "n_sites",
        "n_months",
        "n_features",
        "event_prevalence",
        "missing_pm25",
        "missing_no2",
        "missing_humidity",
    ],
    "value": [
        len(df),
        df["site_id"].nunique(),
        df["month_index"].nunique(),
        len(FEATURE_COLUMNS),
        float(y.mean()),
        int(df["pm25"].isna().sum()),
        int(df["no2"].isna().sum()),
        int(df["humidity"].isna().sum()),
    ],
})

display(summary)
display(df.head())

## Cell 7 — Visualize spatial structure and seasonality

These plots help verify that the synthetic data include a spatial exposure pattern and time variation before applying spatial CV.

In [ ]:
site_mean = (
    df.groupby("site_id")
    .agg(x=("x", "first"), y=("y", "first"), pm25=("pm25", "mean"), event_rate=(TARGET_COLUMN, "mean"))
    .reset_index()
)

fig, ax = plt.subplots(figsize=(6, 5))
sc = ax.scatter(site_mean["x"], site_mean["y"], c=site_mean["pm25"], s=80)
ax.set_title("Synthetic monitoring sites colored by mean PM2.5")
ax.set_xlabel("x coordinate")
ax.set_ylabel("y coordinate")
plt.colorbar(sc, ax=ax, label="Mean PM2.5")
plt.tight_layout()
plt.savefig(Path(OUTPUT_DIR) / "spatial_pm25_map.png", dpi=300)
plt.show()

monthly = df.groupby("month_index").agg(event_rate=(TARGET_COLUMN, "mean"), pm25=("pm25", "mean")).reset_index()
fig, ax = plt.subplots(figsize=(7, 4))
monthly.plot(x="month_index", y="event_rate", marker="o", ax=ax)
ax.set_title("Monthly respiratory-event rate")
ax.set_xlabel("Month index")
ax.set_ylabel("Event rate")
plt.tight_layout()
plt.savefig(Path(OUTPUT_DIR) / "monthly_event_rate.png", dpi=300)
plt.show()

## Cell 8 — Fold-local model factory

The imputer is inside the sklearn pipeline, so missing-value preprocessing is fitted only on each fold's training data. TrustCV controls the fold boundaries.

In [ ]:
def make_model():
    return make_pipeline(
        SimpleImputer(strategy="median"),
        RandomForestClassifier(
            n_estimators=N_ESTIMATORS,
            random_state=RANDOM_STATE,
            class_weight="balanced",
            n_jobs=-1,
        ),
    )

make_model()

## Cell 9 — Build TrustCV spatial splitters and method-specific `split_kwargs`

Each row stores the real TrustCV splitter plus the metadata that its `.split()` method needs. These metadata are later passed to `UniversalCVRunner.run(..., split_kwargs=...)`, not to the model.

In [ ]:
spatial_methods = {
    "SpatialBlockCV": {
        "splitter": SpatialBlockCV(
            n_splits=N_SPATIAL_SPLITS,
            block_shape="grid",
            random_state=RANDOM_STATE,
        ),
        "split_kwargs": {"coordinates": coordinates},
        "description": "Spatial grid blocking using monitoring-site coordinates.",
    },
    "BufferedSpatialCV": {
        "splitter": BufferedSpatialCV(
            n_splits=N_SPATIAL_SPLITS,
            buffer_size=BUFFER_SIZE,
            block_shape="grid",
            random_state=RANDOM_STATE,
        ),
        "split_kwargs": {"coordinates": coordinates},
        "description": "Spatial blocks with a buffer zone around the test block.",
    },
    "SpatiotemporalBlockCV": {
        "splitter": SpatiotemporalBlockCV(
            n_spatial_blocks=N_SPATIAL_BLOCKS,
            n_temporal_blocks=N_TEMPORAL_BLOCKS,
            buffer_space=SPATIOTEMPORAL_BUFFER_SPACE,
            buffer_time=0,
            block_shape="grid",
            random_state=RANDOM_STATE,
        ),
        "split_kwargs": {"coordinates": coordinates, "timestamps": timestamps},
        "description": "Joint spatial and temporal blocking using coordinates and monthly timestamps.",
    },
    "EnvironmentalHealthCV": {
        "splitter": EnvironmentalHealthCV(
            spatial_blocks=N_SPATIAL_BLOCKS,
            temporal_strategy="seasonal",
            environmental_vars=["pm25", "no2"],
            buffer_config=ENV_BUFFER_CONFIG,
        ),
        "split_kwargs": {
            "coordinates": coordinates,
            "timestamps": timestamps,
            "environmental_data": environmental_data,
        },
        "description": "Environmental-health split using spatial, seasonal, and exposure metadata.",
    },
}

method_rows = []
for method_name, cfg in spatial_methods.items():
    splitter = cfg["splitter"]
    split_kwargs = cfg["split_kwargs"]
    try:
        n_splits = splitter.get_n_splits(X, y, groups)
    except TypeError:
        n_splits = splitter.get_n_splits()
    method_rows.append({
        "Method": method_name,
        "TrustCV splitter": splitter.__class__.__name__,
        "Estimated n_splits": n_splits,
        "split_kwargs passed to splitter only": ", ".join(split_kwargs.keys()),
        "Description": cfg["description"],
    })
method_table = pd.DataFrame(method_rows)
display(method_table)

## Cell 10 — Split diagnostics before modeling

Before training, this cell calls each TrustCV splitter with its own `split_kwargs`. The diagnostics check fold sizes, held-out sites, shared sites, spatial separation, and validation prevalence.

In [ ]:
def min_train_test_distance(train_coordinates, test_coordinates):
    """Compute the minimum Euclidean train-test distance without needing scipy."""
    diff = train_coordinates[:, None, :] - test_coordinates[None, :, :]
    distances = np.sqrt(np.sum(diff ** 2, axis=2))
    return float(np.min(distances)) if distances.size else np.nan


def split_diagnostics(method_name, splitter, split_kwargs):
    rows = []
    for fold_id, (train_idx, test_idx) in enumerate(
        splitter.split(X, y=y, groups=groups, **split_kwargs),
        start=1,
    ):
        rows.append({
            "method": method_name,
            "fold": fold_id,
            "n_train": len(train_idx),
            "n_test": len(test_idx),
            "train_sites": len(np.unique(groups[train_idx])),
            "test_sites": len(np.unique(groups[test_idx])),
            "shared_sites": len(set(groups[train_idx]) & set(groups[test_idx])),
            "min_train_test_distance": min_train_test_distance(coordinates[train_idx], coordinates[test_idx]),
            "test_prevalence": float(y[test_idx].mean()),
        })
    return pd.DataFrame(rows)

split_diagnostic_table = pd.concat(
    [
        split_diagnostics(name, cfg["splitter"], cfg["split_kwargs"])
        for name, cfg in spatial_methods.items()
    ],
    ignore_index=True,
)

display(split_diagnostic_table.groupby("method").agg(
    folds=("fold", "count"),
    mean_train=("n_train", "mean"),
    mean_test=("n_test", "mean"),
    max_shared_sites=("shared_sites", "max"),
    mean_min_distance=("min_train_test_distance", "mean"),
    mean_test_prevalence=("test_prevalence", "mean"),
).reset_index())

## Cell 11 — Run spatial CV through TrustCV UniversalCVRunner

This is the main test of the fixed toolbox API. Each TrustCV splitter is passed directly to `UniversalCVRunner`. The splitter metadata are supplied through `split_kwargs`, and model-fitting metadata are kept separate in `fit_kwargs`. This verifies that coordinates and environmental metadata do not leak into `model.fit()`.

In [ ]:
def run_spatial_method(method_name, splitter, split_kwargs):
    print("=" * 80)
    print(method_name)
    print("=" * 80)

    leakage_callback = LeakageDetectionCallback(
        data=(X, y),
        groups=groups,
        timestamps=timestamps,
        coordinates=coordinates,
        verbose=1,
    )
    class_callback = ClassDistributionLogger(
        labels=y,
        label_names={0: "no_event", 1: "respiratory_event"},
        verbose=0,
    )

    runner = UniversalCVRunner(
        cv_splitter=splitter,
        framework="sklearn",
        verbose=1 if SMOKE_MODE else 0,
    )

    cv_result = runner.run(
        model=make_model,
        data=(X, y, groups),
        groups=groups,
        metrics=["accuracy", "balanced_accuracy", "f1", "precision", "recall", "roc_auc"],
        callbacks=[leakage_callback, class_callback],
        split_kwargs=split_kwargs,
        fit_kwargs={},
        evaluate_kwargs={},
    )

    print(cv_result.summary())

    clinical = ClinicalMetrics(confidence_level=0.95, prevalence=float(y.mean()))
    oob_metrics = oob_clinical_metrics(cv_result, y, clinical=clinical)

    return {
        "method": method_name,
        "splitter": splitter,
        "split_kwargs": split_kwargs,
        "cv_result": cv_result,
        "oob_metrics": oob_metrics,
        "leakage_callback": leakage_callback,
    }

runner_outputs = {}
for method_name, cfg in spatial_methods.items():
    runner_outputs[method_name] = run_spatial_method(
        method_name,
        cfg["splitter"],
        cfg["split_kwargs"],
    )

## Cell 12 — OOB clinical metrics from TrustCV results

`oob_clinical_metrics()` reads the stored predictions and probabilities from `CVResults` and sends them to `ClinicalMetrics.calculate_all()`.

In [ ]:
def fmt_ci(value, ci, digits=3):
    if value is None or pd.isna(value):
        return "NA"
    if ci is None:
        return f"{float(value):.{digits}f}"
    return f"{float(value):.{digits}f} ({float(ci[0]):.{digits}f}, {float(ci[1]):.{digits}f})"

rows = []
for method_name, output in runner_outputs.items():
    metrics = output["oob_metrics"] or {}
    rows.append({
        "CV method": method_name,
        "AUC (95% CI)": fmt_ci(metrics.get("auc_roc"), metrics.get("auc_roc_ci")),
        "Average precision": f"{metrics.get('average_precision', np.nan):.3f}",
        "Sensitivity": f"{metrics.get('sensitivity', np.nan):.3f}",
        "Specificity": f"{metrics.get('specificity', np.nan):.3f}",
        "Youden J": f"{metrics.get('youdens_index', np.nan):.3f}",
        "Accuracy": f"{metrics.get('accuracy', np.nan):.3f}",
        "PPV": f"{metrics.get('ppv', np.nan):.3f}",
        "NPV": f"{metrics.get('npv', np.nan):.3f}",
    })

manuscript_table = pd.DataFrame(rows)
display(manuscript_table)

## Cell 13 — Leakage and spatial-proximity audit with TrustCV DataLeakageChecker

This audit uses the same TrustCV splitter and the same `split_kwargs` used by the runner. It checks the first fold for grouped/site overlap, timestamp overlap, and coordinate-aware leakage reports.

In [ ]:
def explicit_leakage_audit(method_name, splitter, split_kwargs):
    checker = DataLeakageChecker(verbose=False)
    train_idx, test_idx = next(iter(splitter.split(X, y=y, groups=groups, **split_kwargs)))
    report = checker.check_cv_splits(
        X_train=X[train_idx],
        X_test=X[test_idx],
        y_train=y[train_idx],
        y_test=y[test_idx],
        patient_ids_train=groups[train_idx],
        patient_ids_test=groups[test_idx],
        timestamps_train=timestamps[train_idx],
        timestamps_test=timestamps[test_idx],
        coordinates_train=coordinates[train_idx],
        coordinates_test=coordinates[test_idx],
    )
    return {
        "CV method": method_name,
        "has_leakage": bool(report.has_leakage),
        "severity": report.severity,
        "leakage_types": ", ".join(report.leakage_types) if report.leakage_types else "none",
        "shared_sites": len(set(groups[train_idx]) & set(groups[test_idx])),
    }

leakage_audit_table = pd.DataFrame([
    explicit_leakage_audit(name, cfg["splitter"], cfg["split_kwargs"])
    for name, cfg in spatial_methods.items()
])
display(leakage_audit_table)

## Cell 14 — Fold-level table from CVResults

This cell confirms that `UniversalCVRunner` produced fold-level scores for each method.

In [ ]:
fold_rows = []
for method_name, output in runner_outputs.items():
    cv_result = output["cv_result"]
    for fold_id, ((train_idx, test_idx), score_dict) in enumerate(zip(cv_result.indices, cv_result.scores), start=1):
        fold_rows.append({
            "CV method": method_name,
            "fold": fold_id,
            "n_train": len(train_idx),
            "n_test": len(test_idx),
            "accuracy": score_dict.get("accuracy", np.nan),
            "balanced_accuracy": score_dict.get("balanced_accuracy", np.nan),
            "f1": score_dict.get("f1", np.nan),
            "roc_auc": score_dict.get("roc_auc", np.nan),
        })

fold_result_table = pd.DataFrame(fold_rows)
display(fold_result_table.head())
display(fold_result_table.groupby("CV method")[["accuracy", "balanced_accuracy", "f1", "roc_auc"]].mean().reset_index())

## Cell 15 — Visualize UniversalCVRunner outputs

These figures summarize the TrustCV OOB clinical metrics and split diagnostics.

In [ ]:
plot_rows = []
for method_name, output in runner_outputs.items():
    m = output["oob_metrics"] or {}
    plot_rows.append({
        "CV method": method_name,
        "AUC": m.get("auc_roc", np.nan),
        "Average precision": m.get("average_precision", np.nan),
    })
plot_df = pd.DataFrame(plot_rows).set_index("CV method")

fig, ax = plt.subplots(figsize=(9, 5))
plot_df.plot(kind="bar", ax=ax)
ax.set_ylim(0, 1)
ax.set_title("TrustCV spatial CV: OOB clinical metrics")
ax.set_ylabel("Metric value")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(Path(OUTPUT_DIR) / "spatial_universal_auc_auprc.png", dpi=300)
plt.show()

fig, ax = plt.subplots(figsize=(9, 4))
(
    split_diagnostic_table.groupby("method")["min_train_test_distance"]
    .mean()
    .sort_values()
    .plot(kind="bar", ax=ax)
)
ax.set_title("Mean minimum train-test spatial distance")
ax.set_ylabel("Distance in synthetic coordinate units")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(Path(OUTPUT_DIR) / "spatial_universal_distance_diagnostic.png", dpi=300)
plt.show()

## Cell 16 — Export outputs

The exported files include API inventory, method table, split diagnostics, fold-level CV results, leakage audit, and manuscript-style metrics. These files can be added to the repository or used to check Colab output.

In [ ]:
output_dir = Path(OUTPUT_DIR).resolve()
output_dir.mkdir(parents=True, exist_ok=True)

api_inventory.to_csv(output_dir / "spatial_split_kwargs_trustcv_api_inventory.csv", index=False)
method_table.to_csv(output_dir / "spatial_split_kwargs_method_table.csv", index=False)
split_diagnostic_table.to_csv(output_dir / "spatial_split_kwargs_split_diagnostics.csv", index=False)
fold_result_table.to_csv(output_dir / "spatial_split_kwargs_fold_results.csv", index=False)
leakage_audit_table.to_csv(output_dir / "spatial_split_kwargs_leakage_audit.csv", index=False)
manuscript_table.to_csv(output_dir / "spatial_split_kwargs_manuscript_table.csv", index=False)
with open(output_dir / "spatial_split_kwargs_manuscript_table.md", "w", encoding="utf-8") as f:
    f.write(manuscript_table.to_markdown(index=False))

import shutil
archive_base = output_dir.parent / output_dir.name
zip_path = shutil.make_archive(
    base_name=str(archive_base),
    format="zip",
    root_dir=str(output_dir.parent),
    base_dir=output_dir.name,
)
print("Output directory:", output_dir)
print("Zip file:", zip_path)
print("Files:")
for p in sorted(output_dir.iterdir()):
    print("-", p.name)


## Cell 17 — Interpretation scaffold

After running the notebook, use this text only after replacing the placeholder numbers with the exported table values.

In [ ]:
print("Interpretation scaffold:")
print(
    "The synthetic environmental-health example exercised all four spatial methods "
    "through TrustCV splitters and the updated UniversalCVRunner split_kwargs API. "
    "Spatial metadata were passed only to the splitter via split_kwargs, while the "
    "sklearn estimator received only the fold-local training data. SpatialBlockCV tests "
    "general spatial blocking, BufferedSpatialCV excludes training observations near "
    "the test block, SpatiotemporalBlockCV holds out combined spatial-time blocks, and "
    "EnvironmentalHealthCV combines spatial, seasonal, and exposure-similarity constraints. "
    "OOB clinical metrics were computed from TrustCV CVResults using oob_clinical_metrics "
    "and ClinicalMetrics."
)